# Train And Save SAE From A Trained Transformer

This notebook only does path setup, SAE training, and saving the SAE artifact for later analysis.

In [21]:
import os
import subprocess
from pathlib import Path

import torch

print(f'Current notebook cwd: {Path.cwd()}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device count: {torch.cuda.device_count()}')
    print(f'Current CUDA device: {torch.cuda.get_device_name(0)}')
else:
    print('Warning: CUDA not available, will run on CPU unless you set device manually.')

!nvidia-smi

Current notebook cwd: /home/ponsin
CUDA available: True
CUDA device count: 1
Current CUDA device: NVIDIA H100
Tue Mar 17 17:49:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100                    On  |   00000000:04:00.0 Off |                    0 |
| N/A   30C    P0            114W /  700W |    1327MiB /  95830MiB |      0%      Default |
|                             

## Configure Paths, SAE Hyperparameters, And SAE Data Splits

Use the consolidated transformer output file produced by `main.py` (typically ending with `.pt`, and in your Slurm naming often `.pkl.pt`).
Then choose where to save the trained SAE artifact, plus the train/eval RHM sizes used for SAE training vs later analysis.

In [ ]:
# Required input artifact from transformer training (single consolidated file from main.py)
train_output = Path('/work/pcsl/ponsin/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt')

# Path to the exact train_sae.py script you want to run
train_sae_script = Path('/home/ponsin/SAE-on-RHM/train_sae.py')

# Required output artifacts for the two SAE sets
sae_output_path_all_tokens = Path('/work/pcsl/ponsin/SAE/v_16_L_3_m_4/Trained_SAE_all_tokens_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1_sae.pt')
sae_output_path_cls_token = Path('/work/pcsl/ponsin/SAE/v_16_L_3_m_4/Trained_SAE_cls_token_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1_sae.pt')

# SAE training settings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sae_layers = 'all'        # e.g. 'all' or '0,1,2'
sae_latent_dim = None     # None means 4 * embedding_dim
sae_lambda_l1 = 3
sae_lr = 5e-5
sae_steps = 2**13
sae_sample_batch_size = 2**7    # number of RHM samples per transformer forward
sae_batch_limit = 0           # 0 means no cap on activations used per SAE step
sae_print_freq = 2**8

# RHM sizes/seeds for SAE pipeline
sae_train_size = 2**14           # data used to train SAE
sae_eval_size = 2**14             # separate data reserved for later SAE analysis
sae_train_seed_sample = None      # None => transformer seed + 1 (different data, same rules)
sae_eval_seed_sample = None       # None => SAE train seed + 1

print(f'train_output: {train_output}')
print(f'train_sae_script: {train_sae_script}')
print(f'sae_output_path_all_tokens: {sae_output_path_all_tokens}')
print(f'sae_output_path_cls_token: {sae_output_path_cls_token}')
print(f'sae_sample_batch_size: {sae_sample_batch_size}')
print(f'sae_batch_limit: {sae_batch_limit}')
print(f'sae_train_size: {sae_train_size}')
print(f'sae_eval_size: {sae_eval_size}')
print(f'sae_train_seed_sample: {sae_train_seed_sample}')
print(f'sae_eval_seed_sample: {sae_eval_seed_sample}')

train_output: /work/pcsl/ponsin/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt
train_sae_script: /home/ponsin/SAE-on-RHM/train_sae.py
sae_output_path_all_tokens: /work/pcsl/ponsin/SAE/v_16_L_3_m_4/Trained_SAE_all_tokens_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1_sae.pt
sae_output_path_cls_token: /work/pcsl/ponsin/SAE/v_16_L_3_m_4/Trained_SAE_cls_token_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1_sae.pt
sae_sample_batch_size: 128
sae_batch_limit: 0
sae_train_size: 16384
sae_eval_size: 16384
sae_train_seed_sample: None
sae_eval_seed_sample: None


In [49]:
def _assert_exists(path_obj, name):
    if path_obj is None:
        raise ValueError(f'{name} is None')
    if not Path(path_obj).exists():
        raise FileNotFoundError(f'{name} not found: {path_obj}')

_assert_exists(train_output, 'train_output')
_assert_exists(train_sae_script, 'train_sae_script')

# Ensure this artifact carries the exact RHM rules used in transformer training.
blob = torch.load(train_output, map_location='cpu')
if not isinstance(blob, dict) or 'output' not in blob:
    raise ValueError(
        'train_output must be a consolidated main.py artifact containing an output dict.'
    )
if 'rules' not in blob['output']:
    raise ValueError(
        'train_output is missing output.rules. Re-run transformer training with the updated Sbatch_trsf_for_SAE.sh.'
    )
print('Verified train_output contains output.rules for fixed-RHM SAE training.')

def build_sae_cmd(sae_output_path, activation_source):
    cmd = [
        'python', str(train_sae_script),
        '--train_output', str(train_output),
        '--outname', str(sae_output_path),
        '--device', device,
        '--sae_layers', str(sae_layers),
        '--sae_lambda_l1', str(float(sae_lambda_l1)),
        '--sae_lr', str(float(sae_lr)),
        '--sae_steps', str(int(sae_steps)),
        '--sae_sample_batch_size', str(int(sae_sample_batch_size)),
        '--sae_batch_limit', str(int(sae_batch_limit)),
        '--sae_activation_source', str(activation_source),
        '--sae_print_freq', str(int(sae_print_freq)),
        '--sae_train_size', str(int(sae_train_size)),
        '--sae_eval_size', str(int(sae_eval_size)),
    ]

    if sae_latent_dim is not None:
        cmd += ['--sae_latent_dim', str(int(sae_latent_dim))]
    if sae_train_seed_sample is not None:
        cmd += ['--sae_train_seed_sample', str(int(sae_train_seed_sample))]
    if sae_eval_seed_sample is not None:
        cmd += ['--sae_eval_seed_sample', str(int(sae_eval_seed_sample))]

    return cmd

cmd_all_tokens = build_sae_cmd(sae_output_path_all_tokens, 'all_tokens')
cmd_cls_token = build_sae_cmd(sae_output_path_cls_token, 'cls_token')

print('Command (all_tokens):')
print(' '.join(cmd_all_tokens))
print('')
print('Command (cls_token):')
print(' '.join(cmd_cls_token))

Verified train_output contains output.rules for fixed-RHM SAE training.
Command (all_tokens):
python /home/ponsin/SAE-on-RHM/train_sae.py --train_output /work/pcsl/ponsin/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt --outname /work/pcsl/ponsin/SAE/v_16_L_3_m_4/Trained_SAE_all_tokens_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1_sae.pt --device cuda --sae_layers all --sae_lambda_l1 3.0 --sae_lr 5e-07 --sae_steps 8192 --sae_sample_batch_size 128 --sae_batch_limit 0 --sae_activation_source all_tokens --sae_print_freq 256 --sae_train_size 16384 --sae_eval_size 16384

Command (cls_token):
python /home/ponsin/SAE-on-RHM/train_sae.py --train_output /work/pcsl/ponsin/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt --outname /work/pcsl/ponsin/SAE/v_16_L_3_m_4/Trained_SAE_cls_token_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1_sae.pt --device cuda --s

/tmp/2654298/ipykernel_1542924/356620418.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = torch.load(train_output, map_location='cpu')


In [50]:
def run_and_check(cmd, label):
    print(f'Running SAE training: {label}')
    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'{label} failed with return code {result.returncode}')
    print(f'{label} finished successfully.')
    return result.stdout

sae_train_logs = {}
sae_train_logs['all_tokens'] = run_and_check(cmd_all_tokens, 'all_tokens SAE')
print('')
sae_train_logs['cls_token'] = run_and_check(cmd_cls_token, 'cls_token SAE')

print('Both SAE trainings finished successfully.')

Running SAE training: all_tokens SAE
SAE data split: train_size=16384 (seed_sample=826178), eval_size=16384 (seed_sample=826179)
Transformer training seed_sample=826177
Using fixed RHM rules loaded from training artifact.
# parameters: 9465872
sae layer 0 step 0/8192 total=3736.989990 recon=3736.258545 sparse=0.243776 active_fraction=0.502424 dead_features=0
sae layer 0 step 256/8192 total=3602.639404 recon=3601.604004 sparse=0.345111
sae layer 0 step 512/8192 total=3510.875000 recon=3509.384766 sparse=0.496783
sae layer 0 step 768/8192 total=3189.898926 recon=3187.914062 sparse=0.661611
sae layer 0 step 1024/8192 total=2968.738525 recon=2966.201172 sparse=0.845778
sae layer 0 step 1280/8192 total=2711.614014 recon=2708.466064 sparse=1.049344
sae layer 0 step 1536/8192 total=2469.927002 recon=2466.293945 sparse=1.211059
sae layer 0 step 1792/8192 total=2239.948730 recon=2235.963623 sparse=1.328348
sae layer 0 step 2048/8192 total=2118.453857 recon=2114.081543 sparse=1.457404
sae layer 

KeyboardInterrupt: 

In [45]:
for label, path in [('all_tokens SAE', sae_output_path_all_tokens), ('cls_token SAE', sae_output_path_cls_token)]:
    p = Path(path)
    print(f'{label} output path: {p.resolve()}')
    if not p.exists():
        raise FileNotFoundError(f'Could not find {label} output at {p}')

print('Both SAE artifacts are ready for loading in a later step.')

all_tokens SAE output path: /work/pcsl/ponsin/SAE/v_16_L_3_m_4/Trained_SAE_all_tokens_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1_sae.pt
cls_token SAE output path: /work/pcsl/ponsin/SAE/v_16_L_3_m_4/Trained_SAE_cls_token_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1_sae.pt
Both SAE artifacts are ready for loading in a later step.


In [47]:
# Compact SAE analysis on a fresh evaluation split
import copy
import sys
from pathlib import Path

import torch

repo_root = Path(train_sae_script).resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import init
import models
from datasets.random_hierarchy_model import sample_trees

# -----------------------------
# Analysis configuration
# -----------------------------
analysis_eval_size = 2**15
analysis_eval_seed_sample = 98765432
analysis_batch_size = 256
analysis_device = device


def load_transformer_and_loader(train_output_path, eval_size, eval_seed, batch_size, device_name):
    blob = torch.load(train_output_path, map_location='cpu')
    if not isinstance(blob, dict) or 'config' not in blob or 'output' not in blob:
        raise ValueError('train_output must contain config and output dictionaries.')
    if 'model' not in blob['output'] or 'rules' not in blob['output'] or blob['output']['rules'] is None:
        raise ValueError('train_output must contain output.model and output.rules.')

    cfg = copy.deepcopy(blob['config'])
    model_state = blob['output']['model']
    rules = blob['output']['rules']

    trees_eval = sample_trees(
        num_data=int(eval_size),
        rules=rules,
        prior=None,
        probs=None,
        seed=int(eval_seed),
    )
    inputs_eval = trees_eval[cfg.num_layers]
    targets_eval = trees_eval[0]

    data_cfg = copy.deepcopy(cfg)
    data_cfg.train_size = int(eval_size)
    data_cfg.test_size = 0
    data_cfg.batch_size = max(1, min(int(batch_size), int(eval_size)))
    loader, _ = init.init_data(inputs_eval, targets_eval, data_cfg)

    model = init.init_model(cfg)
    model.load_state_dict(model_state)
    model = model.to(device_name).eval()
    for p in model.parameters():
        p.requires_grad = False

    return model, loader


def load_sae_set(ckpt_path, input_dim, device_name):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    if 'sae_state' not in ckpt or 'sae_layers' not in ckpt:
        raise ValueError(f'Invalid SAE checkpoint format: {ckpt_path}')

    sae_state = ckpt['sae_state']
    sae_layers = [int(x) for x in ckpt['sae_layers']]
    sae_metrics = ckpt.get('sae_metrics', {})

    modules = {}
    for layer in sae_layers:
        state = sae_state[layer] if layer in sae_state else sae_state[str(layer)]
        latent_dim = None
        if layer in sae_metrics and isinstance(sae_metrics[layer], dict) and 'latent_dim' in sae_metrics[layer]:
            latent_dim = int(sae_metrics[layer]['latent_dim'])
        elif str(layer) in sae_metrics and isinstance(sae_metrics[str(layer)], dict) and 'latent_dim' in sae_metrics[str(layer)]:
            latent_dim = int(sae_metrics[str(layer)]['latent_dim'])
        if latent_dim is None:
            latent_dim = int(state['encoder.weight'].shape[0])

        sae = models.SparseAutoencoder(input_dim=input_dim, latent_dim=latent_dim).to(device_name)
        sae.load_state_dict(state)
        sae.eval()
        for p in sae.parameters():
            p.requires_grad = False
        modules[layer] = sae

    return modules, sae_layers


def compute_activity_stats(model, loader, sae_modules, sae_layers, mode, device_name):
    assert mode in {'all_tokens', 'cls_token'}
    ever_active = {l: torch.zeros(sae_modules[l].latent_dim, dtype=torch.bool, device=device_name) for l in sae_layers}
    active_sum = {l: 0.0 for l in sae_layers}
    active_count = {l: 0 for l in sae_layers}

    buffers = {l: [] for l in sae_layers}
    hooks = []
    for l in sae_layers:
        hooks.append(model.blocks[l].register_forward_hook(lambda _m, _i, o, layer=l: buffers[layer].append(o.detach())))

    with torch.no_grad():
        for x_batch, _ in loader:
            _ = model(x_batch.to(device_name))
            for l in sae_layers:
                if not buffers[l]:
                    continue
                act = buffers[l].pop(0)
                act = act[:, :1, :] if mode == 'cls_token' else act[:, 1:, :]
                act = act.reshape(-1, act.size(-1))
                if act.numel() == 0:
                    continue
                _, z = sae_modules[l](act)
                is_active = z > 0
                active_sum[l] += is_active.float().sum().item()
                active_count[l] += is_active.numel()
                ever_active[l] |= is_active.any(dim=0)

    for h in hooks:
        h.remove()

    stats = {}
    for l in sae_layers:
        dead = int((~ever_active[l]).sum().item())
        latent = int(sae_modules[l].latent_dim)
        stats[l] = {
            'latent_dim': latent,
            'dead_features': dead,
            'dead_feature_ratio': dead / max(latent, 1),
            'mean_active_feature_ratio': active_sum[l] / max(active_count[l], 1),
        }
    return stats


def print_stats(title, stats):
    print('=' * 88)
    print(title)
    print('-' * 88)
    print(f"{'layer':>6} | {'latent_dim':>10} | {'dead_features':>13} | {'dead_ratio':>10} | {'mean_active_ratio':>17}")
    print('-' * 88)
    for l in sorted(stats.keys()):
        s = stats[l]
        print(f"{l:6d} | {s['latent_dim']:10d} | {s['dead_features']:13d} | {s['dead_feature_ratio']:10.6f} | {s['mean_active_feature_ratio']:17.6f}")


for p in [train_output, sae_output_path_all_tokens, sae_output_path_cls_token]:
    if not Path(p).exists():
        raise FileNotFoundError(f'Missing required artifact: {p}')

model, eval_loader = load_transformer_and_loader(
    train_output_path=train_output,
    eval_size=analysis_eval_size,
    eval_seed=analysis_eval_seed_sample,
    batch_size=analysis_batch_size,
    device_name=analysis_device,
)

all_modules, all_layers = load_sae_set(sae_output_path_all_tokens, model.embedding_dim, analysis_device)
cls_modules, cls_layers = load_sae_set(sae_output_path_cls_token, model.embedding_dim, analysis_device)

all_stats = compute_activity_stats(model, eval_loader, all_modules, all_layers, 'all_tokens', analysis_device)
cls_stats = compute_activity_stats(model, eval_loader, cls_modules, cls_layers, 'cls_token', analysis_device)

print(f'Analysis eval size: {analysis_eval_size}, seed: {analysis_eval_seed_sample}, batch_size: {analysis_batch_size}')
print_stats('All-tokens SAE stats', all_stats)
print_stats('CLS-token SAE stats', cls_stats)

/tmp/2654298/ipykernel_1542924/249344188.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = torch.load(train_output_path, map_location='cpu')
/tmp/2654298/ipykernel

# parameters: 9465872
Analysis eval size: 32768, seed: 98765432, batch_size: 256
All-tokens SAE stats
----------------------------------------------------------------------------------------
 layer | latent_dim | dead_features | dead_ratio | mean_active_ratio
----------------------------------------------------------------------------------------
     0 |       2048 |             0 |   0.000000 |          0.504505
     1 |       2048 |             0 |   0.000000 |          0.727058
     2 |       2048 |           299 |   0.145996 |          0.821547
CLS-token SAE stats
----------------------------------------------------------------------------------------
 layer | latent_dim | dead_features | dead_ratio | mean_active_ratio
----------------------------------------------------------------------------------------
     0 |       2048 |           308 |   0.150391 |          0.539696
     1 |       2048 |             0 |   0.000000 |          0.649658
     2 |       2048 |           454 |  